In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.streaming.checkpoints;

In [0]:
# dbutils.fs.rm("/Volumes/workspace/streaming/checkpoints/ecommerce_pipeline/bronze", True)
# dbutils.fs.rm("/Volumes/workspace/streaming/checkpoints/ecommerce_pipeline/silver", True)
# dbutils.fs.rm("/Volumes/workspace/streaming/checkpoints/ecommerce_pipeline/gold", True)

In [0]:
# %sql
# drop table if exists workspace.streaming.ecommerce_bronze;
# drop table if exists workspace.streaming.ecommerce_silver;
# drop table if exists workspace.streaming.ecommerce_gold_scd2;


In [0]:
from pyspark.sql.functions import col, current_timestamp

In [0]:
from config.kafka_secrets import KAFKA_BOOTSTRAP_SERVERS, KAFKA_API_KEY, KAFKA_API_SECRET


api_key = KAFKA_API_KEY

api_secret = KAFKA_API_SECRET

bootstrap_server = KAFKA_BOOTSTRAP_SERVERS


In [0]:
from pyspark.sql.functions import current_timestamp

# Step 1: Read from Kafka (Confluent Cloud)

jaas_config = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";'

df_kafka = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_server)
    .option("subscribe", "ecommerce_v4")
    .option("startingOffsets", "earliest")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", jaas_config)
    .load()
)

# Step 2: Convert binary → string (same as your code)
df_bronze = df_kafka.selectExpr(
    "CAST(key AS STRING) as key",
    "CAST(value AS STRING) as value",
    "topic",
    "partition",
    "offset",
    "timestamp as kafka_timestamp",
)

df_bronze = df_bronze.withColumn("ingestion_time", current_timestamp())


# Step 3: Write to Delta (Bronze table)
query = (
    df_bronze.writeStream.format("delta")
    .outputMode("append")
    .option("mergeSchema", "true")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/streaming/checkpoints/ecommerce_pipeline/bronze",
    )
    .trigger(availableNow=True)
    .table("workspace.streaming.ecommerce_bronze")
)

# query.awaitTermination()

In [0]:
# %sql
# select * from workspace.streaming.ecommerce_bronze

In [0]:
%sql
select count(*) from workspace.streaming.ecommerce_bronze